# Notebook 01 — MediaPipe + 눈/입 추적 기초

## 목표
- MediaPipe Face Mesh로 얼굴 랜드마크 추출
- EAR (Eye Aspect Ratio) — 눈 감김 감지
- MAR (Mouth Aspect Ratio) — 하품 감지
- Head Pose Estimation — 고개 방향 감지
- 웹캠 실시간 테스트

In [ ]:
# 한글 폰트 설정
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False
print('폰트 설정 완료')

In [ ]:
# 패키지 설치 확인
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

try:
    import mediapipe
    print(f'mediapipe {mediapipe.__version__} 설치됨')
except ImportError:
    print('mediapipe 설치 중...')
    install('mediapipe')

try:
    import cv2
    print(f'opencv {cv2.__version__} 설치됨')
except ImportError:
    print('opencv 설치 중...')
    install('opencv-python')

import numpy as np
from scipy.spatial import distance
print('모든 패키지 준비 완료')

## 1. MediaPipe Face Mesh 기초

Face Mesh는 얼굴에서 **468개 랜드마크**를 실시간으로 추출합니다.

DMS에서 핵심 랜드마크:
- **눈**: 왼쪽 33~133번, 오른쪽 362~263번
- **입**: 61, 291, 13, 14번 등
- **코끝**: 1번 (Head Pose 기준점)

In [ ]:
import mediapipe as mp
import cv2
import numpy as np

# MediaPipe 초기화
mp_face_mesh = mp.solutions.face_mesh
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

# 눈 랜드마크 인덱스 (MediaPipe 기준)
# 왼쪽 눈
LEFT_EYE = [362, 385, 387, 263, 373, 380]
# 오른쪽 눈
RIGHT_EYE = [33, 160, 158, 133, 153, 144]
# 입
MOUTH = [61, 291, 13, 14, 17, 0, 402, 178]

print('MediaPipe Face Mesh 초기화 완료')
print(f'왼쪽 눈 랜드마크: {LEFT_EYE}')
print(f'오른쪽 눈 랜드마크: {RIGHT_EYE}')

## 2. EAR (Eye Aspect Ratio) — 눈 감김 감지

```
EAR = (|p2-p6| + |p3-p5|) / (2 × |p1-p4|)
```

- **EAR > 0.25**: 눈 뜸 (정상)
- **EAR < 0.25**: 눈 감김 (졸음 의심)
- **연속 15프레임 이상** EAR < 0.25: 졸음 경고

In [ ]:
from scipy.spatial import distance as dist

def calculate_EAR(eye_landmarks):
    """
    EAR (Eye Aspect Ratio) 계산
    eye_landmarks: [(x1,y1), (x2,y2), ..., (x6,y6)] — 6개 좌표
    """
    # 수직 거리 2개
    A = dist.euclidean(eye_landmarks[1], eye_landmarks[5])
    B = dist.euclidean(eye_landmarks[2], eye_landmarks[4])
    # 수평 거리 1개
    C = dist.euclidean(eye_landmarks[0], eye_landmarks[3])
    
    ear = (A + B) / (2.0 * C)
    return ear

# 테스트: 눈 뜬 상태 시뮬레이션
eye_open = [(0,0), (1,2), (2,2), (3,0), (2,-2), (1,-2)]
ear_open = calculate_EAR(eye_open)
print(f'눈 뜬 상태 EAR: {ear_open:.3f}')  # 약 0.5 이상

# 눈 감은 상태 시뮬레이션
eye_closed = [(0,0), (1,0.1), (2,0.1), (3,0), (2,-0.1), (1,-0.1)]
ear_closed = calculate_EAR(eye_closed)
print(f'눈 감은 상태 EAR: {ear_closed:.3f}')  # 약 0.1 이하

## 3. MAR (Mouth Aspect Ratio) — 하품 감지

```
MAR = (|p2-p8| + |p3-p7| + |p4-p6|) / (2 × |p1-p5|)
```

- **MAR > 0.6**: 하품 (입 크게 벌림)
- EAR 낮음 + MAR 높음 동시 → 졸음 확실

In [ ]:
def calculate_MAR(mouth_landmarks):
    """
    MAR (Mouth Aspect Ratio) 계산
    mouth_landmarks: 8개 좌표
    """
    # 수직 거리 3개
    A = dist.euclidean(mouth_landmarks[1], mouth_landmarks[7])
    B = dist.euclidean(mouth_landmarks[2], mouth_landmarks[6])
    C = dist.euclidean(mouth_landmarks[3], mouth_landmarks[5])
    # 수평 거리 1개
    D = dist.euclidean(mouth_landmarks[0], mouth_landmarks[4])
    
    mar = (A + B + C) / (2.0 * D)
    return mar

print('MAR 함수 정의 완료')
print('임계값: MAR > 0.6 → 하품 감지')

## 4. Head Pose Estimation — 고개 방향 감지

3D 얼굴 모델과 2D 랜드마크를 매핑해서 고개 각도(pitch/yaw/roll)를 추정합니다.

- **Yaw > ±30°**: 좌우로 고개 돌림 (딴 곳 봄)
- **Pitch > 20°**: 고개 숙임 (졸음)
- **Roll > ±20°**: 고개 기울임

In [ ]:
def get_head_pose(landmarks, image_shape):
    """
    Head Pose Estimation
    solvePnP로 3D 회전 벡터 → pitch/yaw/roll 각도 반환
    """
    h, w = image_shape[:2]
    
    # 3D 얼굴 기준 좌표 (표준 모델)
    face_3d_model = np.array([
        [0.0, 0.0, 0.0],          # 코끝 (1번)
        [0.0, -330.0, -65.0],     # 턱 (152번)
        [-225.0, 170.0, -135.0],  # 왼쪽 눈 코너 (33번)
        [225.0, 170.0, -135.0],   # 오른쪽 눈 코너 (263번)
        [-150.0, -150.0, -125.0], # 왼쪽 입 (61번)
        [150.0, -150.0, -125.0],  # 오른쪽 입 (291번)
    ], dtype=np.float64)
    
    # 2D 랜드마크 추출 (인덱스: 1, 152, 33, 263, 61, 291)
    key_indices = [1, 152, 33, 263, 61, 291]
    face_2d = np.array([
        [landmarks[i].x * w, landmarks[i].y * h]
        for i in key_indices
    ], dtype=np.float64)
    
    # 카메라 내부 파라미터 (근사값)
    focal_length = w
    cam_matrix = np.array([
        [focal_length, 0, w / 2],
        [0, focal_length, h / 2],
        [0, 0, 1]
    ], dtype=np.float64)
    dist_coeffs = np.zeros((4, 1))
    
    # PnP 풀기
    success, rot_vec, trans_vec = cv2.solvePnP(
        face_3d_model, face_2d, cam_matrix, dist_coeffs
    )
    
    if not success:
        return None, None, None
    
    # 회전 벡터 → 회전 행렬 → 오일러 각도
    rot_matrix, _ = cv2.Rodrigues(rot_vec)
    angles, _, _, _, _, _ = cv2.RQDecomp3x3(rot_matrix)
    
    pitch = angles[0] * 360  # 상하
    yaw = angles[1] * 360    # 좌우
    roll = angles[2] * 360   # 기울임
    
    return pitch, yaw, roll

print('Head Pose Estimation 함수 정의 완료')
print('임계값:')
print('  Yaw > ±30°  → 전방 주시 이탈')
print('  Pitch > 20° → 고개 숙임 (졸음)')
print('  Roll > ±20° → 고개 기울임')

## 5. 통합 함수 — 이미지 1장에서 모든 지표 추출

In [ ]:
def extract_driver_features(image, face_mesh):
    """
    이미지 1장에서 운전자 상태 지표 전부 추출
    
    Returns:
        dict: {
            'ear': float,       # 눈 감김 정도 (0~1)
            'mar': float,       # 입 벌림 정도 (0~1)
            'pitch': float,     # 고개 상하 각도
            'yaw': float,       # 고개 좌우 각도
            'roll': float,      # 고개 기울임 각도
            'detected': bool    # 얼굴 감지 여부
        }
    """
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb)
    
    if not results.multi_face_landmarks:
        return {'detected': False, 'ear': None, 'mar': None,
                'pitch': None, 'yaw': None, 'roll': None}
    
    landmarks = results.multi_face_landmarks[0].landmark
    h, w = image.shape[:2]
    
    # 눈 좌표 추출
    left_eye_pts = [(landmarks[i].x * w, landmarks[i].y * h) for i in LEFT_EYE]
    right_eye_pts = [(landmarks[i].x * w, landmarks[i].y * h) for i in RIGHT_EYE]
    
    # 입 좌표 추출
    mouth_pts = [(landmarks[i].x * w, landmarks[i].y * h) for i in MOUTH]
    
    # EAR 계산 (양쪽 평균)
    left_ear = calculate_EAR(left_eye_pts)
    right_ear = calculate_EAR(right_eye_pts)
    ear = (left_ear + right_ear) / 2.0
    
    # MAR 계산
    mar = calculate_MAR(mouth_pts)
    
    # Head Pose 계산
    pitch, yaw, roll = get_head_pose(landmarks, image.shape)
    
    return {
        'detected': True,
        'ear': round(ear, 4),
        'mar': round(mar, 4),
        'pitch': round(pitch, 2) if pitch is not None else None,
        'yaw': round(yaw, 2) if yaw is not None else None,
        'roll': round(roll, 2) if roll is not None else None,
    }

print('통합 특징 추출 함수 정의 완료')

## 6. 웹캠 실시간 테스트

실제 웹캠에서 실시간으로 EAR/MAR/Head Pose를 시각화합니다.

**종료**: `q` 키를 누르세요.

In [ ]:
# 임계값 설정
EAR_THRESHOLD = 0.25    # 눈 감김 기준
MAR_THRESHOLD = 0.6     # 하품 기준
YAW_THRESHOLD = 30.0    # 고개 좌우 기준
PITCH_THRESHOLD = 20.0  # 고개 숙임 기준
DROWSY_FRAMES = 15      # 연속 프레임 수 기준

def draw_status(frame, features, frame_counter, alert_level):
    """화면에 상태 정보 표시"""
    h, w = frame.shape[:2]
    
    # 배경 박스
    cv2.rectangle(frame, (0, 0), (300, 160), (0, 0, 0), -1)
    
    if not features['detected']:
        cv2.putText(frame, 'Face Not Detected', (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        return frame
    
    # 지표 표시
    ear_color = (0, 0, 255) if features['ear'] < EAR_THRESHOLD else (0, 255, 0)
    mar_color = (0, 0, 255) if features['mar'] > MAR_THRESHOLD else (0, 255, 0)
    
    cv2.putText(frame, f"EAR: {features['ear']:.3f}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, ear_color, 2)
    cv2.putText(frame, f"MAR: {features['mar']:.3f}", (10, 55),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, mar_color, 2)
    
    if features['yaw'] is not None:
        yaw_color = (0, 0, 255) if abs(features['yaw']) > YAW_THRESHOLD else (0, 255, 0)
        cv2.putText(frame, f"Yaw: {features['yaw']:.1f}°", (10, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, yaw_color, 2)
        cv2.putText(frame, f"Pitch: {features['pitch']:.1f}°", (10, 105),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    
    # 경고 표시
    alert_colors = {0: (0, 255, 0), 1: (0, 165, 255), 2: (0, 0, 255)}
    alert_texts = {0: 'NORMAL', 1: 'WARNING', 2: 'DANGER!'}
    cv2.putText(frame, alert_texts[alert_level], (10, 140),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, alert_colors[alert_level], 2)
    
    return frame


def run_webcam_test():
    """웹캠 실시간 DMS 테스트"""
    cap = cv2.VideoCapture(0)
    drowsy_counter = 0
    distracted_counter = 0
    
    with mp_face_mesh.FaceMesh(
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as face_mesh:
        
        print('웹캠 시작 — q 키로 종료')
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            
            frame = cv2.flip(frame, 1)  # 거울 모드
            features = extract_driver_features(frame, face_mesh)
            
            alert_level = 0
            
            if features['detected']:
                # 졸음 카운터
                if features['ear'] < EAR_THRESHOLD or features['mar'] > MAR_THRESHOLD:
                    drowsy_counter += 1
                else:
                    drowsy_counter = max(0, drowsy_counter - 1)
                
                # 부주의 카운터 (고개 방향)
                if features['yaw'] and abs(features['yaw']) > YAW_THRESHOLD:
                    distracted_counter += 1
                else:
                    distracted_counter = max(0, distracted_counter - 1)
                
                # 경고 레벨 결정
                if drowsy_counter > DROWSY_FRAMES * 2 or distracted_counter > DROWSY_FRAMES * 2:
                    alert_level = 2  # 위험
                elif drowsy_counter > DROWSY_FRAMES or distracted_counter > DROWSY_FRAMES:
                    alert_level = 1  # 주의
            
            frame = draw_status(frame, features, drowsy_counter, alert_level)
            cv2.imshow('DMS - Driver Monitoring System', frame)
            
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    
    cap.release()
    cv2.destroyAllWindows()
    print('웹캠 종료')

# 실행 (웹캠이 연결된 경우)
run_webcam_test()

## 7. 결과 시각화 — EAR/MAR 변화 그래프

실제 영상 없이도 알고리즘을 시각적으로 확인할 수 있습니다.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 시뮬레이션 데이터 생성 (실제 웹캠 대신)
np.random.seed(42)
frames = np.arange(200)

# 정상 → 졸음 → 정상 패턴 시뮬레이션
ear_values = np.concatenate([
    np.random.normal(0.32, 0.02, 60),   # 정상
    np.random.normal(0.18, 0.03, 60),   # 졸음
    np.random.normal(0.31, 0.02, 80),   # 회복
])
mar_values = np.concatenate([
    np.random.normal(0.2, 0.05, 80),    # 정상
    np.random.normal(0.7, 0.08, 30),    # 하품
    np.random.normal(0.2, 0.05, 90),    # 정상
])
yaw_values = np.concatenate([
    np.random.normal(0, 5, 120),        # 전방 주시
    np.random.normal(40, 5, 40),        # 부주의
    np.random.normal(0, 5, 40),         # 복귀
])

fig, axes = plt.subplots(3, 1, figsize=(12, 8))
fig.suptitle('운전자 상태 모니터링 지표', fontsize=14, fontweight='bold')

# EAR 그래프
axes[0].plot(frames, ear_values, 'b-', linewidth=1.5, label='EAR')
axes[0].axhline(y=0.25, color='r', linestyle='--', label='임계값 (0.25)')
axes[0].fill_between(frames, ear_values, 0.25,
                     where=(ear_values < 0.25), alpha=0.3, color='red', label='졸음 구간')
axes[0].set_ylabel('EAR')
axes[0].set_title('눈 감김 감지 (EAR)')
axes[0].legend()
axes[0].set_ylim(0, 0.5)

# MAR 그래프
axes[1].plot(frames, mar_values, 'g-', linewidth=1.5, label='MAR')
axes[1].axhline(y=0.6, color='r', linestyle='--', label='임계값 (0.6)')
axes[1].fill_between(frames, mar_values, 0.6,
                     where=(mar_values > 0.6), alpha=0.3, color='orange', label='하품 구간')
axes[1].set_ylabel('MAR')
axes[1].set_title('하품 감지 (MAR)')
axes[1].legend()
axes[1].set_ylim(0, 1.0)

# Yaw 그래프
axes[2].plot(frames, yaw_values, 'purple', linewidth=1.5, label='Yaw')
axes[2].axhline(y=30, color='r', linestyle='--', label='임계값 (±30°)')
axes[2].axhline(y=-30, color='r', linestyle='--')
axes[2].fill_between(frames, yaw_values, 30,
                     where=(yaw_values > 30), alpha=0.3, color='red', label='부주의 구간')
axes[2].set_ylabel('각도 (°)')
axes[2].set_xlabel('프레임')
axes[2].set_title('전방 주시 이탈 감지 (Yaw)')
axes[2].legend()

plt.tight_layout()
plt.savefig('../output/01_dms_signals.png', dpi=150, bbox_inches='tight')
plt.show()
print('그래프 저장 완료: output/01_dms_signals.png')

## 정리

| 지표 | 임계값 | 의미 |
|------|--------|------|
| EAR | < 0.25 | 눈 감김 |
| MAR | > 0.60 | 하품 |
| Yaw | > ±30° | 고개 좌우 이탈 |
| Pitch | > 20° | 고개 숙임 |
| PERCLOS | > 15% | 단위시간 내 졸음 비율 |

**다음 Notebook 02**: 이 지표들을 조합해서 졸음/부주의 판단 로직 + YOLOv8 휴대폰 감지 추가